In [8]:
from tarfile import DEFAULT_FORMAT

from tqdm.notebook import tqdm
import os,pickle,glob,gc,sys
import polars as pl
import cudf,itertools
import pandas as pd
import numpy as np
print(cudf.__version__)

25.12.00


In [19]:
train=pd.read_parquet('/home/mingyu/Recommand-System/项目/OTTO/data/processData/train.parquet')
test=pd.read_parquet('/home/mingyu/Recommand-System/项目/OTTO/data/processData/test.parquet')

sample_sub=pd.read_csv('/home/mingyu/Recommand-System/项目/OTTO/data/processData/sample_submission.csv')

In [22]:
train.index=pd.MultiIndex.from_frame(train[['session']])
test.index=pd.MultiIndex.from_frame(test[['session']])

In [23]:
data=pd.concat([train,test])
data.head(),data.shape

(         session      aid          ts  type
 session                                    
 0              0  1517085  1659304800     0
 0              0  1563459  1659304904     0
 0              0  1309446  1659367439     0
 0              0    16246  1659367719     0
 0              0  1781822  1659367871     0,
 (223644219, 4))

In [24]:
from collections import defaultdict,Counter
next_AIDs=defaultdict(Counter)
chunk_size=30_000
max_ts=data['ts'].max()
min_ts=data['ts'].min()
nextID=defaultdict(Counter)

In [25]:
sessions=data.session.unique()

In [ ]:
for i in range(0,sessions.shape[0],chunk_size):
    # reset_index(drop=True)重建行索引，丢弃旧索引
    current_chunk=data.loc[sessions[i]:sessions[min(sessions.shape[0]-1,i+chunk_size-1)]].reset_index(drop=True)
    # 取出每个session后30个，考虑性能和最近行为
    # as_index保持原有的索引，不用session做索引
    current_chunk=current_chunk.groupby('session',as_index=False).nth(list(range(-30,0))).reset_index(drop=True)
    # 按照session连接，对同一个session内的所有行（不区分属性）做笛卡尔积
    consecutive_AIDs=current_chunk.merge(current_chunk,on='session')
    # 排除自己和自己
    consecutive_AIDs=consecutive_AIDs[consecutive_AIDs.aid_x!=consecutive_AIDs.aid_y]
    # 通过时间戳(>=0)的方式保证两两算一次，不考虑顺序
    consecutive_AIDs['days_elapsed']=(consecutive_AIDs.ts_y-consecutive_AIDs.ts_x)/(24*60*60)
    consecutive_AIDs=consecutive_AIDs[(consecutive_AIDs.days_elapsed>=0) & (consecutive_AIDs.days_elapsed<=1)]

    for aid_x,aid_y in zip(consecutive_AIDs['aid_x'],consecutive_AIDs['aid_y']):
        next_AIDs[aid_x][aid_y]+=1

In [ ]:
session_types=['clicks','carts','orders']
# 聚合session
test_session_AIDs=test.reset_index(drop=True).groupby('session')['aid'].apply(list)
test_session_types=test.reset_index(drop=True).groupby('session')['type'].apply(list)

labels=[]

no_data=0 # 输出的预选共现商品为空
no_data_all_aids=0 #无法通过共现矩阵补充预选商品
type_weight_multipliers={0:1,1:6,2:3}

for AIDs,types in zip(test_session_AIDs,test_session_types):
    # 当session的操作数>=20，对操作加权，权重由位置和类型决定
    if len(AIDs)>=15:
        # base底数，endpoint是否包含右边界
        weights=np.logspace(0.1,1,len(AIDs),base=2,endpoint=True)-1
        aids_temp=defaultdict(lambda: 0)
        for aid,w,t in zip(AIDs,weights,types):
            aids_temp[aid]+=w*type_weight_multipliers[t]
        sorted_aids=[k for k,v in sorted(aids_temp.items(),key=lambda item:-item[1])]
        labels.append(sorted_aids[:15])
    else:
        # 对短序列用共现矩阵补全
        AIDs=list(dict.fromkeys(AIDs[::-1])) # 去重,倒序保证能保留最近的操作
        AIDs_len_start=len(AIDs)
        candidates=[]
        # 为每个商品找到出现次数最多的20个
        for AID in AIDs:
            if AID in next_AIDs:
                # most_common返回出现次数最多的元素及其频次
                candidates+=[aid for aid,count in next_AIDs[AID].most_common(20)]
        # 保证不重复添加到最终结果中
        AIDs+=[AID for AID ,cnt in Counter(candidates).most_common(40) if AID not in AIDs]

        labels.append(AIDs[:15])
        if candidates==[]:no_data+=1
        if AIDs_len_start==len(AIDs):no_data_all_aids+=1
print(no_data_all_aids,no_data)